# Supervised Learning
Sentiment analysis, star prediction, category detection, and model comparisons.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from transformers import pipeline

df = pd.read_csv('../data/dataset_enriched.csv')
# Keep it small for quick execution
df = df.sample(5000, random_state=42).dropna(subset=['avis', 'note'])

## 1. Sentiment Analysis (TF-IDF + Logistic Regression)
We map notes to sentiments: 1-2 (Neg), 3 (Neu), 4-5 (Pos)

In [ ]:
def map_sentiment(n):
    if n <= 2: return 0
    elif n == 3: return 1
    else: return 2

df['sentiment'] = df['note'].apply(map_sentiment)
X_train, X_test, y_train, y_test = train_test_split(df['avis'].astype(str), df['sentiment'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=3000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

lr = LogisticRegression(max_iter=500)
lr.fit(X_train_vec, y_train)
y_pred = lr.predict(X_test_vec)
acc_sentiment = accuracy_score(y_test, y_pred)
print("Sentiment Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

## 2. Star Prediction (1-5 stars)
Using Random Forest on TF-IDF.

In [ ]:
y_train_stars, y_test_stars = train_test_split(df['note'], test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train_vec, y_train_stars)
y_pred_stars = rf.predict(X_test_vec)
acc_stars = accuracy_score(y_test_stars, y_pred_stars)
print(f"Star Prediction Accuracy: {acc_stars:.2f}")

## 3. Zero-Shot Classification (Categories)
Using HuggingFace to detect topics for a few reviews.

In [ ]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)
labels = ["Pricing", "Customer Service", "Claims", "Coverage"]

sample_text = "Je paie vraiment trop cher pour cette assurance auto, c'est inacceptable."
res = classifier(sample_text, labels)

print(f"Text: {sample_text}")
print("Predicted Categories & Scores:")
for label, score in zip(res['labels'], res['scores']):
    print(f"- {label}: {score:.3f}")

## 4. Model Comparison & Error Analysis

In [ ]:
results = pd.DataFrame({
    'Model': ['TF-IDF + LR (Sentiment)', 'TF-IDF + RF (Stars)', 'BERT Zero-Shot (Categories)'],
    'Accuracy / Metric': [acc_sentiment, acc_stars, 'N/A (Unsupervised)']
})
display(results)

print("""
### Error Analysis
- **Sarcasm and Irony**: The model struggles identifying fake positive feedback like "Bravo pour l'attente" (Well done for the wait).
- **Short Texts**: Reviews with just 1 or 2 words (e.g., "Moyen") don't carry enough TF-IDF weight and are often misclassified.
- **Class Imbalance**: Positive ratings often outnumber negative ones, which biases the standard logistic regression slightly towards predicting 4s and 5s.
""")